# Cohort A: build the patient-episode feature table

Starting from `cohort_a_derived.cohort_a_source` (built in `01_set_up_cohort_a_cartesian`), this
notebook:

1. Restricts to the 11 CIPOC cancer types and flags registry/EHR match quality.
2. Builds a 6-month post-diagnosis "episode" window per patient.
3. Derives treatment-intensity features from that window: contact days, visit regularity, and
   radiation / surgery / chemotherapy procedure and drug-order counts.
4. Saves the result as `cohort_a_derived.cohort_a_feature_table`.

In [ ]:
USE CATALOG your_catalog;

Quick sanity check: what registry site values exist in the source table?

In [ ]:
select distinct registry_site from cohort_a_derived.cohort_a_source

### Map EHR cancer categories to CIPOC registry site codes
The registry and the EHR use different label sets for the same cancer types (fuzzy matches between
`ehr_rollup2` and `registry_site`). This view adds `ehr_rollup2_mod`, translating each EHR rollup
category into the corresponding registry site code, and restricts to the 11 CIPOC cancer types.

In [ ]:
--swap in matching cancer names. they are fuzzy matches.
create or replace view matching_cancers as
select a.*,
case when ehr_rollup2 = 'Leukemia' then 'LEUK'
when ehr_rollup2 = 'Colon and rectum' then 'COLORECTAL'
when ehr_rollup2 = 'Digestive system' then 'LIVER&BILE'
when ehr_rollup2 = 'Respiratory system' then 'LUNG&BRONCHUS'
when ehr_rollup2 = 'Lymphoma' then 'NON_HODG'
when ehr_rollup2 = 'Myeloma' then 'MYELOMA'
when ehr_rollup2 = 'Male genital system' then 'PROSTATE'
when ehr_rollup2 = 'Breast' then 'BREAST'
when ehr_rollup2 = 'Female genital system' then 'UTERINE|OVARY'
when ehr_rollup2 = 'Urinary system' then 'URINARY BLADDER'
else 'problem' end as ehr_rollup2_mod
from cohort_a_derived.cohort_a_source a
WHERE ehr_rollup2 IN ('Leukemia','Colon and rectum','Digestive system','Respiratory system','Lymphoma','Myeloma','Male genital system','Breast','Female genital system','Urinary system')

### Flag registry/EHR match quality
For patients found in both the registry and the EHR, flags whether the diagnosis dates are within
365 days of each other and the cancer types agree (`'ok'`) or not (`'bad match'`). Rows that are
registry-only (no matching EHR episode) are dropped here.

In [ ]:
--flag registry/ehr matches that don't seem to be good quality
--limit ehr rows to the 11 CIPOC cancers
--automatically drops the reg-only people
drop view if exists filtered_cases;
create view filtered_cases as
select *,
case when abs(daysbt) <= 365 and ehr_rollup2_mod = registry_site then 'ok'
when abs(daysbt) <= 365 and ehr_rollup2_mod = 'UTERINE|OVARY' and registry_site IN ('OVARY','UTERINE') then 'ok'
when abs(daysbt) > 365 or ehr_rollup2_mod <> registry_site then 'bad match'
else 'problem' end as match_quality
from matching_cancers
where match_type = 'Registry and EHR'

### Restrict to analytic registry cases
Cleans out registry rows with malformed diagnosis dates, keeps only "analytic" cases
(`CLASS_OF_CASE_N610` between 0 and 22), and joins to the quality-flagged matches from the previous
cell, keeping only `'ok'` matches. 

In [ ]:
--filter down further to analytic cases, just for training purposes
create or replace view analytic_cases as
with bad_dates as (
select row_num
from cohort_a.crstar_pk n
where length(DATE_OF_DIAGNOSIS_N390) <> 8
)
,

crstar_clean as (
select * from cohort_a.crstar_pk where row_num NOT IN (select * from bad_dates)
)
,

analytics as (
SELECT * 
from crstar_clean
WHERE CLASS_OF_CASE_N610 between 0 and 22
)

select * from filtered_cases f JOIN analytics a ON f.registry_person_id = cast(a.person_id as string) and to_date(CAST(DATE_OF_DIAGNOSIS_N390 AS STRING), 'yyyyMMdd') = f.registry_cancer_dx_date and f.match_quality = 'ok'

### Build the 6-month post-diagnosis contact window
For each analytic case, looks at all visits overlapping a 180-day window after diagnosis and merges
overlapping/adjacent visits into contiguous "islands" of contact, then sums total contact days per
episode. Also computes `years_w_unc`, the patient's total span of UNC visit history in years.

In [ ]:
drop view if exists full_patient_list_filtered;
create temp view full_patient_list_filtered as

with preagg as (
    select *, ehr_episode_start as period_start, date_add(ehr_episode_start,180) as period_end from analytic_cases),

--this query basically creates mini episodes for each patient out of the visits  
clipped as (
    select 
    p.*,
    greatest(v.visit_start_date, period_start) as admit_date, 
    least(v.visit_end_date, period_end) as discharge_date
    from preagg p 
    left join cohort_a.visit_occurrence v ON p.ehr_person_id = v.person_id
    where v.VISIT_START_DATE <= p.period_end AND v.VISIT_END_DATE >= period_start
),

-- this query creates a boolean var that is basically like a true or false answering the question: does this new visit occur after the end of the previous visit? 1 = true
flagged as (
    select *,
    case when admit_date > max(discharge_date) over (partition by ehr_person_id order by admit_date rows between unbounded preceding and 1 preceding) then 1 else 0 
    end as is_new_visit
    from clipped c
), 
-- this query groups all overlapping visits into islands 
-- and the boolean var is a true or false of this question: is this visit on the previous island or is it a new island 

visit_island as ( 
    select *,
    sum(is_new_visit) over (partition by ehr_person_id order by admit_date) as visit_island_id 
    from flagged
    ),
  --then I add it all up :) and back to the previous code we were using 
preagg2 as(
   select ehr_person_id, ehr_episode_start, ehr_episode_end, ehr_rollup2_mod, registry_person_id, registry_cancer_dx_date, registry_site, match_type, daysbt, match_quality, sum(visit_days) as total_contact_days 
from 
(select ehr_person_id, ehr_episode_start, ehr_episode_end, ehr_rollup2_mod, registry_person_id, registry_cancer_dx_date, registry_site, match_type, daysbt, match_quality, visit_island_id, 
date_diff(max(discharge_date), min((admit_date))) + 1 as visit_days 
from visit_island 
group by ehr_person_id,ehr_episode_start,ehr_episode_end, ehr_rollup2_mod, registry_person_id, registry_cancer_dx_date, registry_site, match_type, daysbt, match_quality, visit_island_id) as per_visit_island
group by ehr_person_id,ehr_episode_start, ehr_episode_end, ehr_rollup2_mod, registry_person_id, registry_cancer_dx_date, registry_site, match_type, daysbt, match_quality),

preagg3 as (
  select p.ehr_person_id, p.ehr_episode_start, p.ehr_episode_end, p.ehr_rollup2_mod, p.registry_person_id, p.registry_cancer_dx_date, p.registry_site, p.match_type, p.daysbt, p.match_quality, total_contact_days, max(v.visit_start_date) as maxunc, min(v.visit_start_date) as minunc
  from preagg2 p JOIN cohort_a.visit_occurrence v ON p.ehr_person_id = v.person_id
  group by p.ehr_person_id, p.ehr_episode_start, p.ehr_episode_end, p.ehr_rollup2_mod, p.registry_person_id, p.registry_cancer_dx_date, p.registry_site, p.match_type, p.daysbt, p.match_quality, total_contact_days
)

select ehr_person_id, ehr_episode_start, ehr_episode_end, ehr_rollup2_mod as ehr_rollup2, registry_person_id, registry_cancer_dx_date, registry_site, match_type, daysbt, match_quality, total_contact_days, date_diff(maxunc, minunc)/365.25 as years_w_unc
from preagg3

Adds `avg_days_bt_visits`: the average gap between visits in the 6-month post-diagnosis window (a visit-regularity feature).

In [ ]:
drop view if exists full_patient_list_filtered_lag;
create temp view full_patient_list_filtered_lag as

--calculate avg time between visits in 6 mo after dx
with one as (
select distinct f.*, v.visit_start_date
from full_patient_list_filtered f JOIN cohort_a.visit_occurrence v ON f.ehr_person_id = v.person_id and  v.visit_start_date between f.ehr_episode_start and date_add(f.ehr_episode_start,180)
),

lagfunction as (
select distinct f.*, visit_start_date,
datediff(visit_start_date,lag(visit_start_date) OVER (PARTITION BY ehr_person_id, ehr_rollup2 ORDER BY visit_start_date)) AS days_bt_visits
from one f 
)

select ehr_person_id, ehr_episode_start, ehr_episode_end, ehr_rollup2, registry_person_id, registry_cancer_dx_date, registry_site, match_type, daysbt, match_quality, total_contact_days, years_w_unc, avg(days_bt_visits) as avg_days_bt_visits
from lagfunction
group by ehr_person_id, ehr_episode_start, ehr_episode_end, ehr_rollup2, registry_person_id, registry_cancer_dx_date, registry_site, match_type, daysbt, match_quality, total_contact_days, years_w_unc

Adds `rad_days`: count of distinct dates with a radiation procedure in the 6-month window (procedure codes defined in `reference.radsurgchemocodes`).

In [ ]:
--create radiation feature
--get any radiation that happened within six months of episode start
drop view if exists full_patient_list_filtered_rad;
create temp view full_patient_list_filtered_rad as
with preaggproc as (
select distinct f.*, p.procedure_concept_id, p.procedure_date
from cohort_a.procedure_occurrence p JOIN reference.radsurgchemocodes r ON r.standard_concept_id = p.procedure_concept_id and lower(r.concept_type) = 'radiation' RIGHT JOIN full_patient_list_filtered_lag f ON f.ehr_person_id = p.person_id and (p.procedure_date between f.ehr_episode_start and date_add(f.ehr_episode_start,180) or p.procedure_date is null))

select ehr_person_id, ehr_episode_start, ehr_episode_end, ehr_rollup2, registry_person_id, registry_cancer_dx_date, registry_site, match_type, daysbt, match_quality, total_contact_days, years_w_unc, avg_days_bt_visits, count(distinct procedure_date) as rad_days
from preaggproc
group by ehr_person_id, ehr_episode_start, ehr_episode_end, ehr_rollup2, registry_person_id, registry_cancer_dx_date, registry_site, match_type, daysbt, match_quality, total_contact_days, years_w_unc, avg_days_bt_visits

Adds `surg_days`: count of distinct dates with a surgery procedure in the same window.

In [ ]:
--create surgery feature
--get any surgery that happened within six months of episode start
drop view if exists full_patient_list_filtered_rad_surg;
create temp view full_patient_list_filtered_rad_surg as
with preaggproc as (
select distinct f.*, p.procedure_concept_id, p.procedure_date
from cohort_a.procedure_occurrence p JOIN reference.radsurgchemocodes r ON r.standard_concept_id = p.procedure_concept_id and lower(r.concept_type) = 'surgery' RIGHT JOIN full_patient_list_filtered_rad f ON f.ehr_person_id = p.person_id and (p.procedure_date between f.ehr_episode_start and date_add(f.ehr_episode_start,180) or p.procedure_date is null))

select ehr_person_id, ehr_episode_start, ehr_episode_end, ehr_rollup2, registry_person_id, registry_cancer_dx_date, registry_site, match_type, daysbt, match_quality, total_contact_days, years_w_unc, avg_days_bt_visits, rad_days, count(distinct procedure_date) as surg_days
from preaggproc
group by ehr_person_id, ehr_episode_start, ehr_episode_end, ehr_rollup2, registry_person_id, registry_cancer_dx_date, registry_site, match_type, daysbt, match_quality, total_contact_days, years_w_unc, avg_days_bt_visits, rad_days

Adds `chemproc_days`: count of distinct dates with a chemotherapy procedure in the same window.

In [ ]:
--create chemo proc feature
--get any chemo proc that happened within six months of episode start
drop view if exists full_patient_list_filtered_rad_surg_chemproc;
create temp view full_patient_list_filtered_rad_surg_chemproc as
with preaggproc as (
select distinct f.*, p.procedure_concept_id, p.procedure_date
from cohort_a.procedure_occurrence p JOIN reference.radsurgchemocodes r ON r.standard_concept_id = p.procedure_concept_id and lower(r.concept_type) = 'chemo' RIGHT JOIN full_patient_list_filtered_rad_surg f ON f.ehr_person_id = p.person_id and (p.procedure_date between f.ehr_episode_start and date_add(f.ehr_episode_start,180) or p.procedure_date is null))

select ehr_person_id, ehr_episode_start, ehr_episode_end, ehr_rollup2, registry_person_id, registry_cancer_dx_date, registry_site, match_type, daysbt, match_quality, total_contact_days, years_w_unc, avg_days_bt_visits, rad_days, surg_days, count(distinct procedure_date) as chemproc_days
from preaggproc
group by ehr_person_id, ehr_episode_start, ehr_episode_end, ehr_rollup2, registry_person_id, registry_cancer_dx_date, registry_site, match_type, daysbt, match_quality, total_contact_days, years_w_unc, avg_days_bt_visits, rad_days, surg_days

Adds `chemdrug_inst`: count of distinct dates with a chemotherapy drug order in the same window.

In [ ]:
--find chemo drug orders within six months of dx date
drop view if exists full_patient_list_filtered_rad_surg_chemproc_drug;
create temp view full_patient_list_filtered_rad_surg_chemproc_drug as
with preaggproc as (
select distinct f.*, p.drug_concept_id, p.drug_exposure_start_date
from cohort_a.drug_exposure p JOIN reference.radsurgchemocodes r ON r.standard_concept_id = p.drug_concept_id and lower(r.concept_type) = 'chemo' RIGHT JOIN full_patient_list_filtered_rad_surg_chemproc f ON f.ehr_person_id = p.person_id and (p.drug_exposure_start_date between f.ehr_episode_start and date_add(f.ehr_episode_start,180) or p.drug_exposure_start_date is null))

select ehr_person_id, ehr_episode_start, ehr_episode_end, ehr_rollup2, registry_person_id, registry_cancer_dx_date, registry_site, match_type, daysbt, match_quality, total_contact_days, years_w_unc, avg_days_bt_visits, rad_days, surg_days, chemproc_days, count(distinct drug_exposure_start_date) as chemdrug_inst
from preaggproc
group by ehr_person_id, ehr_episode_start, ehr_episode_end, ehr_rollup2, registry_person_id, registry_cancer_dx_date, registry_site, match_type, daysbt, match_quality, total_contact_days, years_w_unc, avg_days_bt_visits, rad_days, surg_days, chemproc_days

Recomputes the ICD-10-CM "C" code mapping (same logic as in `01_set_up_cohort_a_cartesian`, rebuilt here as `all_c_codes_in_data` for use in the next cell).

In [ ]:
--convert condition_concept_ids that are actually in the data back to ICD-10s
drop view if exists all_c_codes_in_data;
create view all_c_codes_in_data as 
SELECT distinct c.concept_id as original_concept_id, c2.*
FROM cohort_a.concept c JOIN cohort_a.condition_occurrence co ON c.concept_id = co.condition_concept_id 
    JOIN cohort_a.concept_relationship cr ON c.concept_id = cr.concept_id_2 and relationship_id = 'Maps to'
    JOIN cohort_a.concept c2 ON c2.concept_id = cr.concept_id_1 and c2.vocabulary_id = 'ICD10CM' and c2.concept_code LIKE 'C%' and c2.invalid_reason is null

Adds `ccode_days`: count of distinct dates with any cancer ("C"-coded) diagnosis in the same window.

In [ ]:
--get a count of C codes in the six months post-dx
drop view if exists full_patient_list_filtered_rad_surg_chemproc_drug_dx;
create temp view full_patient_list_filtered_rad_surg_chemproc_drug_dx as
with preaggproc as (
select distinct f.*, co.condition_concept_id, co.condition_start_date
from cohort_a.condition_occurrence co JOIN all_c_codes_in_data ac ON co.condition_concept_id = ac.original_concept_id
RIGHT JOIN full_patient_list_filtered_rad_surg_chemproc_drug f ON f.ehr_person_id = co.person_id and (co.condition_start_date between f.ehr_episode_start and date_add(f.ehr_episode_start,180) or co.condition_start_date is null))

select ehr_person_id, ehr_episode_start, ehr_episode_end, ehr_rollup2, registry_person_id, registry_cancer_dx_date, registry_site, match_type, daysbt, match_quality, total_contact_days, years_w_unc, avg_days_bt_visits, rad_days, surg_days, chemproc_days, chemdrug_inst, count(distinct condition_start_date) as ccode_days
from preaggproc
group by ehr_person_id, ehr_episode_start, ehr_episode_end, ehr_rollup2, registry_person_id, registry_cancer_dx_date, registry_site, match_type, daysbt, match_quality, total_contact_days, years_w_unc, avg_days_bt_visits, rad_days, surg_days, chemproc_days, chemdrug_inst

### Final Cohort A feature table
Combines all the counts above into treatment-intensity ratios (`ccode_ratio`, `treatment_ratio`)
and saves the result as `cohort_a_derived.cohort_a_feature_table`. Drops any episode without a
computable `avg_days_bt_visits`.

In [ ]:
--create final feature table
drop table if exists cohort_a_derived.cohort_a_feature_table;
create table cohort_a_derived.cohort_a_feature_table as
select distinct ehr_person_id, ehr_episode_start, registry_site as reg_cx_type, ehr_rollup2 as ehr_cx_type, match_type, total_contact_days, years_w_unc, avg_days_bt_visits, rad_days, surg_days, chemproc_days, chemdrug_inst, ccode_days, ccode_days/total_contact_days as ccode_ratio, (surg_days + rad_days + chemproc_days + chemdrug_inst)/total_contact_days as treatment_ratio
from full_patient_list_filtered_rad_surg_chemproc_drug_dx 
where avg_days_bt_visits is not null